# GoEmotions Multi-Label Classification with RoBERTa

Clean Trainer-based implementation for easy hyperparameter tuning.

Task: Multi-label emotion classification.


In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset


## Load Dataset

In [ ]:
df = pd.read_csv('goemotions_cleaned.csv')

# This cleaned file includes metadata columns. Keep only text + label columns.
# Label columns start at 'admiration' and are 0/1 indicators.
label_start = df.columns.get_loc('admiration')
emotion_cols = list(df.columns[label_start:])
df = df[['text'] + emotion_cols]

print('Rows:', len(df))
print('Number of emotions:', len(emotion_cols))


Rows: 58004
Number of emotions: 28


In [ ]:
# Map output indices -> your actual emotion names
id2label = {i: lab for i, lab in enumerate(emotion_cols)}
label2id = {lab: i for i, lab in id2label.items()}

model.config.id2label = id2label
model.config.label2id = label2id

## Train / Validation / Test Split
-  80/10/10

In [ ]:
df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42)
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42)

print(f"Train split rows: {len(df_train)} ({len(df_train)/len(df)*100:.2f}%)")
print(f"Validation split rows: {len(df_val)} ({len(df_val)/len(df)*100:.2f}%)")
print(f"Test split rows: {len(df_test)} ({len(df_test)/len(df)*100:.2f}%)")

train_texts = df_train['text'].tolist()
val_texts = df_val['text'].tolist()
test_texts = df_test['text'].tolist()

y_train = df_train[emotion_cols].values
y_val = df_val[emotion_cols].values
y_test = df_test[emotion_cols].values


Train split rows: 46403 (80.00%)
Validation split rows: 5800 (10.00%)
Test split rows: 5801 (10.00%)


## Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('roberta-base')

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Dataset Class (Trainer Compatible)

In [ ]:
class GoEmotionsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = GoEmotionsDataset(train_encodings, y_train)
val_dataset = GoEmotionsDataset(val_encodings, y_val)
test_dataset = GoEmotionsDataset(test_encodings, y_test)


## Load Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=len(emotion_cols)
)

model.config.problem_type = 'multi_label_classification'


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    return {
        'f1_micro': f1_score(labels, preds, average='micro', zero_division=0),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0)
    }


## Training Configuration

In [ ]:
%pip install -U accelerate

In [ ]:
training_args = TrainingArguments(
    output_dir='./goemotions_roberta',
    num_train_epochs=7,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_micro',
    logging_steps=100,
    warmup_steps=200,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.119517,0.118576,0.276923,0.161881
2,0.111832,0.114206,0.323927,0.207215
3,0.103757,0.114757,0.350524,0.253059
4,0.095588,0.116423,0.384615,0.287109
5,0.083393,0.122274,0.387190,0.302515
6,0.074635,0.127177,0.397882,0.318787
7,0.072491,0.129955,0.392873,0.318498


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=20307, training_loss=0.09877710149683487, metrics={'train_runtime': 2737.2406, 'train_samples_per_second': 118.667, 'train_steps_per_second': 7.419, 'total_flos': 2.137098678205747e+16, 'train_loss': 0.09877710149683487, 'epoch': 7.0})

- The only thing you need to change between runs is the run_name variable at the top — just update it to something descriptive like "run2_epochs5_lr2e5" before each new training run. The comparison section at the bottom will automatically pick up all saved CSVs and print them side by side.

In [ ]:
# Extract and display validation history
log_history = trainer.state.log_history
eval_logs = [log for log in log_history if 'eval_f1_micro' in log]
results_df = pd.DataFrame(eval_logs)[['epoch', 'eval_loss', 'eval_f1_micro', 'eval_f1_macro']]
print(results_df)

# Save to CSV with descriptive name — change this name each run!
run_name = "run1_epochs7_lr3e5_bs16_5epochs"
results_df.to_csv(f"{run_name}.csv", index=False)
print(f"\nSaved to {run_name}.csv")

   epoch  eval_loss  eval_f1_micro  eval_f1_macro
0    1.0   0.118576       0.276923       0.161881
1    2.0   0.114206       0.323927       0.207215
2    3.0   0.114757       0.350524       0.253059
3    4.0   0.116423       0.384615       0.287109
4    5.0   0.122274       0.387190       0.302515
5    6.0   0.127177       0.397882       0.318787
6    7.0   0.129955       0.392873       0.318498

Saved to run1_epochs7_lr3e5_bs16_5epochs.csv


# Step 8 - test with the test dataset

- only proceed when we're ready to test

### won't training again with different number of epochs be bad for the model since the model has already 'seen' the test set?

- yes:  if you look at your test score, tweak something (epochs, learning rate, etc.), retrain, and check test again repeatedly. Each time you peek at the test score and make a decision based on it, you're indirectly "leaking" information from the test set into your choices.

- no: The model never trained on it — it was only used in trainer.evaluate(test_dataset) for read-only scoring, not for updating the model's weights. So it's still a valid final benchmark.

- best practice
1. train
2. check EVAL score
3. adjust hyperparameters (only use this to make decisions)
4. retrain

a. final test score -> run ONCE at the very end when you're done experimenting

In [ ]:
results = trainer.evaluate(test_dataset)
print("Final Test Results:")
print(f"  F1 Micro: {results['eval_f1_micro']:.4f}")
print(f"  F1 Macro: {results['eval_f1_macro']:.4f}")

Final Test Results:
  F1 Micro: 0.4125
  F1 Macro: 0.3280


In [ ]:
import torch

device = next(model.parameters()).device

def predict_emotions(text: str, threshold: float = 0.1):
    model.eval()
    enc = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits

    probs = torch.sigmoid(logits).squeeze().detach().cpu()
    id2label = model.config.id2label

    results = {
        id2label[i]: round(float(probs[i]), 4)
        for i in range(len(probs))
        if probs[i] >= threshold
    }

    return dict(sorted(results.items(), key=lambda x: x[1], reverse=True))


print(predict_emotions("I can't believe how amazing that was!"))
print(predict_emotions("This is so depressing, I just want to go home"))
print(predict_emotions('I was terrified I would fail, but when I saw the results, I just burst out laughing in disbelief.'))
print(predict_emotions('I’m happy you got the job, but I can’t deny it hurts a little too.'))

{'admiration': 0.7345, 'surprise': 0.4571, 'excitement': 0.1528}
{'sadness': 0.8229, 'disappointment': 0.4264, 'desire': 0.1311}
{'surprise': 0.79, 'fear': 0.3323, 'amusement': 0.2358}
{'sadness': 0.8246, 'joy': 0.657, 'disappointment': 0.1684, 'love': 0.1002}
